In [1]:
import pandas as pd
import numpy as np
import joblib, json
from pathlib import Path

DATA = Path("data")
b = joblib.load("models/credit_model.joblib")
model, features, cat_cols, cats = b["model"], b["features"], b["cat_cols"], b["categories"]

test_df = pd.read_parquet(DATA / "test_clean.parquet")
for c in cat_cols:
    test_df[c] = pd.Categorical(test_df[c], categories=cats[c])

preds = pd.read_parquet(DATA / "test_preds.parquet").set_index("SK_ID_CURR")
test_df["p"] = preds.loc[test_df["SK_ID_CURR"], "p_lgb"].values
test_df["band"] = np.select([test_df["p"] >= 0.25, test_df["p"] >= 0.12],
                            ["DECLINE", "REVIEW"], default="APPROVE")

def fmt(v):
    if pd.isna(v):
        return "missing"
    if isinstance(v, (float, np.floating)):
        return round(float(v), 3)
    return str(v)

def to_int(v):
    return None if pd.isna(v) else int(round(float(v)))

def fact_sheet(i):
    r = test_df.loc[i]
    contrib = model.predict(test_df.loc[[i], features], pred_contrib=True)[0, :-1]
    s = pd.Series(contrib, index=features)
    up = s[s > 0].sort_values(ascending=False).head(3)
    down = s[s < 0].sort_values().head(3)
    return {
        "applicant_id": int(r["SK_ID_CURR"]),
        "default_risk_percent": round(float(r["p"]) * 100, 1),
        "recommendation": r["band"],
        "loan_amount": to_int(r["AMT_CREDIT"]),
        "annual_income": to_int(r["AMT_INCOME_TOTAL"]),
        "annual_payment": to_int(r["AMT_ANNUITY"]),
        "risk_increasing_factors": {f: fmt(test_df.loc[i, f]) for f in up.index},
        "risk_reducing_factors":   {f: fmt(test_df.loc[i, f]) for f in down.index},
    }

example = test_df[test_df["band"] == "REVIEW"].sample(1, random_state=1).index[0]
print(json.dumps(fact_sheet(example), indent=2))

{
  "applicant_id": 213580,
  "default_risk_percent": 16.7,
  "recommendation": "REVIEW",
  "loan_amount": 263686,
  "annual_income": 90000,
  "annual_payment": 28107,
  "risk_increasing_factors": {
    "EXT_SOURCE_3": 0.308,
    "EXT_SOURCE_1": 0.196,
    "inst_late_rate": 0.184
  },
  "risk_reducing_factors": {
    "prev_approved_count": 9.0,
    "AMT_CREDIT": 263686.5,
    "DAYS_BIRTH": "-9978"
  }
}


In [2]:
import os
from dotenv import load_dotenv
from google import genai

NAMES = {
    "EXT_SOURCE_1": "External credit score 1 (0 to 1, higher is safer)",
    "EXT_SOURCE_2": "External credit score 2 (0 to 1, higher is safer)",
    "EXT_SOURCE_3": "External credit score 3 (0 to 1, higher is safer)",
    "inst_late_rate": "Share of past instalments paid late",
    "inst_underpaid_rate": "Share of past instalments underpaid",
    "inst_max_days_late": "Most days late on one instalment",
    "prev_approved_count": "Past Home Credit loans approved",
    "prev_refused_count": "Past Home Credit applications refused",
    "prev_refused_rate": "Share of past applications refused",
    "bureau_active_count": "Active loans with other lenders",
    "bureau_total_debt": "Total debt with other lenders",
    "bureau_oldest_loan_days": "Years since first loan with other lenders",
    "AMT_CREDIT": "Loan amount",
    "AMT_GOODS_PRICE": "Price of goods financed",
    "AMT_ANNUITY": "Annual payment",
    "DAYS_BIRTH": "Age in years",
    "DAYS_EMPLOYED": "Years in current job",
    "credit_to_income": "Loan amount divided by income",
    "annuity_to_income": "Annual payment divided by income",
}

def readable_value(f, v):
    if v == "missing":
        return v
    if f in ("DAYS_BIRTH", "DAYS_EMPLOYED", "bureau_oldest_loan_days"):
        return round(-float(v) / 365.25, 1)
    if f.endswith("_rate"):
        return f"{round(float(v) * 100, 1)}%"
    return v

def readable(sheet):
    out = dict(sheet)
    for key in ("risk_increasing_factors", "risk_reducing_factors"):
        out[key] = {NAMES.get(f, f): readable_value(f, v) for f, v in sheet[key].items()}
    return out

PROMPT = """You are a credit analyst assistant. Write a short credit memo (80 to 120 words) for a loan officer.
Rules:
- Use ONLY the facts in the JSON below. Do not add any number, fact, or assumption that is not in it.
- State the recommendation exactly as given: APPROVE, REVIEW, or DECLINE.
- Mention the default risk percent.
- Explain the main risk-increasing and risk-reducing factors in plain English.
- Do not mention gender. Do not suggest age as a reason to decline.
Facts:
{facts}"""

load_dotenv()
client = genai.Client()

facts = readable(fact_sheet(example))
print(json.dumps(facts, indent=2), "\n")

resp = client.models.generate_content(
    model="gemini-flash-latest",
    contents=PROMPT.format(facts=json.dumps(facts, indent=2)))
print(resp.text)

{
  "applicant_id": 213580,
  "default_risk_percent": 16.7,
  "recommendation": "REVIEW",
  "loan_amount": 263686,
  "annual_income": 90000,
  "annual_payment": 28107,
  "risk_increasing_factors": {
    "External credit score 3 (0 to 1, higher is safer)": 0.308,
    "External credit score 1 (0 to 1, higher is safer)": 0.196,
    "Share of past instalments paid late": "18.4%"
  },
  "risk_reducing_factors": {
    "Past Home Credit loans approved": 9.0,
    "Loan amount": 263686.5,
    "Age in years": 27.3
  }
} 

**Credit Memo: Applicant 213580**  
**Recommendation:** REVIEW  

This applicant has a default risk percent of 16.7%. For the requested loan amount of 263686, the annual payment is 28107 against an annual income of 90000.  

The main risk-increasing factors are low external credit scores—external credit score 1 is 0.196 and external credit score 3 is 0.308—alongside an 18.4% share of past instalments paid late.  

Conversely, primary risk-reducing factors include 9.0 past Home 

In [3]:
import re, time

NUM = r"\d[\d,]*\.?\d*"

def allowed_numbers(obj):
    nums = set()
    def walk(x):
        if isinstance(x, dict):
            for k, v in x.items():
                walk(k); walk(v)
        elif isinstance(x, (int, float)) and not isinstance(x, bool):
            nums.add(abs(float(x)))
        elif isinstance(x, str):
            for m in re.findall(NUM, x):
                nums.add(float(m.replace(",", "").rstrip(".")))
    walk(obj)
    return nums

def check_memo(memo, facts):
    allowed = allowed_numbers(facts)
    found = [float(m.replace(",", "").rstrip(".")) for m in re.findall(NUM, memo)]
    invented = [n for n in found
                if not any(abs(n - a) <= max(0.051, 0.001 * a) for a in allowed)]
    bands = set(re.findall(r"\b(APPROVE|REVIEW|DECLINE)\b", memo))
    return invented, bands == {facts["recommendation"]}

sample = pd.concat([test_df[test_df["band"] == b].sample(n, random_state=7)
                    for b, n in [("APPROVE", 7), ("REVIEW", 7), ("DECLINE", 6)]])

results = []
for k, i in enumerate(sample.index, 1):
    facts = readable(fact_sheet(i))
    try:
        memo = client.models.generate_content(
            model="gemini-flash-latest",
            contents=PROMPT.format(facts=json.dumps(facts, indent=2))).text
        invented, band_ok = check_memo(memo, facts)
        error = None
    except Exception as e:
        memo, invented, band_ok, error = None, None, None, str(e)[:120]
    results.append({"applicant_id": facts["applicant_id"], "band": facts["recommendation"],
                    "band_correct": band_ok, "invented_numbers": invented,
                    "memo": memo, "error": error})
    print(f"{k}/20 done", "| ERROR" if error else "")
    time.sleep(7)

res = pd.DataFrame(results)
res.to_parquet(DATA / "memo_eval.parquet", index=False)

ok = res[res["error"].isna()]
print("\nMemos generated:", len(ok), "of 20")
print("Correct recommendation:", int(ok["band_correct"].sum()), "of", len(ok))
print("Memos with any invented number:", int((ok["invented_numbers"].str.len() > 0).sum()), "of", len(ok))
print(ok.loc[ok["invented_numbers"].str.len() > 0, ["applicant_id", "invented_numbers"]])

1/20 done 
2/20 done 
3/20 done 
4/20 done 
5/20 done 
6/20 done 
7/20 done 
8/20 done 
9/20 done 
10/20 done 
11/20 done 
12/20 done 
13/20 done 
14/20 done 
15/20 done 
16/20 done 
17/20 done 
18/20 done 
19/20 done 
20/20 done | ERROR

Memos generated: 19 of 20
Correct recommendation: 19 of 19
Memos with any invented number: 0 of 19
Empty DataFrame
Columns: [applicant_id, invented_numbers]
Index: []


In [4]:
print("Error message:", res.loc[res["error"].notna(), "error"].values, "\n")

for b in ["APPROVE", "REVIEW", "DECLINE"]:
    row = ok[ok["band"] == b].iloc[0]
    i = test_df.index[test_df["SK_ID_CURR"] == row["applicant_id"]][0]
    print("=" * 70)
    print("FACTS:", json.dumps(readable(fact_sheet(i)), indent=1))
    print("\nMEMO:\n", row["memo"])

Error message: <ArrowStringArray>
['429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and ']
Length: 1, dtype: str 

FACTS: {
 "applicant_id": 183055,
 "default_risk_percent": 4.5,
 "recommendation": "APPROVE",
 "loan_amount": 755190,
 "annual_income": 135000,
 "annual_payment": 31995,
 "risk_increasing_factors": {
  "External credit score 3 (0 to 1, higher is safer)": 0.367,
  "Loan amount": 755190.0,
  "Total debt with other lenders": 1317123.0
 },
 "risk_reducing_factors": {
  "External credit score 2 (0 to 1, higher is safer)": 0.646,
  "External credit score 1 (0 to 1, higher is safer)": 0.553,
  "ORGANIZATION_TYPE": "Government"
 }
}

MEMO:
 **Credit Memo: Applicant 183055**

The recommendation is **APPROVE**. The applicant's default risk percent is 4.5%. Financial details include an annual income of 135000, a requested loan amount of 755190, and an annual payment of 31995.

The primary risk-increasing factors are 

## Notebook 09: AI credit memo

For each applicant, I build a fact sheet: risk percent, recommendation band, loan, income, payment, and the top 3 reasons each way from SHAP. Gemini writes a short memo using only that fact sheet.

**Testing for invented facts (20 applicants, 19 memos; one hit the free-tier limit):**
- Recommendation matched the model: 19 of 19
- Memos with any number not in the fact sheet: 0 of 19
- Manual read of 3 memos: no invented claims, only mild adjectives like "weak" or "only"

Zero errors in 19 does not mean the agent never invents. A rough 95% upper bound is 3/19, about 16%. Testing more memos would tighten that.

**Problems found:**
- One memo cited marital status as a reason. Sensitive attributes should be removed from explanations.
- "Loan amount" appears as a risk-raising reason in one memo and risk-lowering in another, which is confusing for a loan officer.